In [ ]:
#Kiểm tra môi trường
import torch
print(torch.__version__, torch.version.cuda)
print("GPU:", torch.cuda.is_available())

2.11.0+cu128 12.8
GPU: True


In [ ]:
#Cài thư viện đồ thị đúng phiên bản torch của Colab
v = torch.__version__.split('+')[0]
c = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse --only-binary=:all: -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
!pip install torch_geometric scipy scikit-learn pandas

import torch_scatter, torch_sparse
print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
torch_scatter: 2.1.2+pt211cu128
torch_sparse: 0.6.18+pt211cu128


In [ ]:
!git clone https://github.com/seongjunyun/Graph_Transformer_Networks.git
!cp Graph_Transformer_Networks/*.py /content/

import os
need = ['main.py','model_gtn.py','model_fastgtn.py','inits.py','utils.py']
missing = [f for f in need if not os.path.exists(f)]
print("Thiếu:", missing if missing else "Không thiếu gì, OK!")

fatal: destination path 'Graph_Transformer_Networks' already exists and is not an empty directory.
Thiếu: Không thiếu gì, OK!


In [ ]:
# Tải MovieLens 100K + tự build đồ thị dị thể
import os, pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

os.makedirs('data_raw', exist_ok=True)
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O data_raw/ml100k.zip
!unzip -o -q data_raw/ml100k.zip -d data_raw/

# ---- Đọc 3 file thô ----
users = pd.read_csv('data_raw/ml-100k/u.user', sep='|', header=None,
                    names=['uid','age','gender','occupation','zip'])
item_cols = ['mid','title','date','vdate','url'] + [f'g{i}' for i in range(19)]
movies = pd.read_csv('data_raw/ml-100k/u.item', sep='|', header=None,
                     names=item_cols, encoding='latin-1')
ratings = pd.read_csv('data_raw/ml-100k/u.data', sep='\t', header=None,
                      names=['uid','mid','rating','ts'])

n_u, n_m, n_g = len(users), len(movies), 19
N = n_u + n_m + n_g
off_m, off_g = n_u, n_u + n_m
print(f"user {n_u} | movie {n_m} | genre {n_g} | tổng {N} node | {len(ratings)} rating")

# ---- Feature (KHÔNG dùng age vì age là nhãn -> tránh leakage) ----
gender_oh = pd.get_dummies(users['gender']).values.astype(np.float32)      # 2 chiều
occ_oh    = pd.get_dummies(users['occupation']).values.astype(np.float32)  # 21 chiều
genre_mat = movies[[f'g{i}' for i in range(19)]].values.astype(np.float32) # 19 chiều
year = pd.to_datetime(movies['date'], format='%d-%b-%Y', errors='coerce').dt.year
year = ((year - year.min()) / (year.max() - year.min())).fillna(0.5).values.astype(np.float32)

D = 23  # dim lớn nhất (user: 2+21), pad tất cả về cùng dim
def pad(x):
    out = np.zeros((x.shape[0], D), dtype=np.float32); out[:, :x.shape[1]] = x; return out

features = np.vstack([pad(np.hstack([gender_oh, occ_oh])),      # user
                      pad(np.hstack([year[:, None], genre_mat])), # movie
                      pad(np.eye(n_g, dtype=np.float32))])        # genre

# ---- 6 loại cạnh ----
like    = ratings[ratings.rating >= 4]   # thích
dislike = ratings[ratings.rating <= 3]   # không thích
def adj(rows, cols):
    return csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N))

u_like, m_like = like.uid.values - 1, like.mid.values - 1 + off_m
u_dis,  m_dis  = dislike.uid.values - 1, dislike.mid.values - 1 + off_m
mg = movies.melt(id_vars='mid', value_vars=[f'g{i}' for i in range(19)])
mg = mg[mg.value == 1]
m_g = mg.mid.values - 1 + off_m
g_g = mg.variable.str[1:].astype(int).values + off_g

edges = [adj(u_like, m_like), adj(m_like, u_like),
         adj(u_dis,  m_dis),  adj(m_dis,  u_dis),
         adj(m_g, g_g),       adj(g_g, m_g)]

# ---- Nhãn: nhóm tuổi 3 lớp (0: <25, 1: 25-34, 2: >=35), chia 40/20/40 ----
y = np.digitize(users['age'].values, [25, 35])
idx = np.arange(n_u)
tr_i, rest_i, tr_y, rest_y = train_test_split(idx, y, train_size=0.4, stratify=y, random_state=42)
va_i, te_i, va_y, te_y = train_test_split(rest_i, rest_y, train_size=0.33, stratify=rest_y, random_state=42)

labels = [np.vstack((tr_i, tr_y)).T.astype(int),
          np.vstack((va_i, va_y)).T.astype(int),
          np.vstack((te_i, te_y)).T.astype(int)]
print("Phân bố 3 lớp tuổi:", np.bincount(y))

os.makedirs('data/MovieLens', exist_ok=True)
with open('data/MovieLens/node_features.pkl', 'wb') as f: pickle.dump(features, f)
with open('data/MovieLens/edges.pkl', 'wb') as f: pickle.dump(edges, f)
with open('data/MovieLens/labels.pkl', 'wb') as f: pickle.dump(labels, f)
print("Đã lưu data/MovieLens/")

user 943 | movie 1682 | genre 19 | tổng 2644 node | 100000 rating
Phân bố 3 lớp tuổi: [234 310 399]
Đã lưu data/MovieLens/


In [ ]:
import pickle
with open('data/MovieLens/node_features.pkl','rb') as f: X = pickle.load(f)
with open('data/MovieLens/edges.pkl','rb') as f: E = pickle.load(f)
with open('data/MovieLens/labels.pkl','rb') as f: L = pickle.load(f)

print("features:", X.shape)                    # (2644, 23)
print("số loại cạnh:", len(E), "| shape:", E[0].shape)  # 6 | (2644, 2644)
print("số cạnh mỗi loại:", [int(e.nnz) for e in E])
print("train/val/test:", [len(l) for l in L])  # ~377 / ~186 / ~380

features: (2644, 23)
số loại cạnh: 6 | shape: (2644, 2644)
số cạnh mỗi loại: [55375, 55375, 44625, 44625, 2893, 2893]
train/val/test: [377, 186, 380]


In [ ]:
with open('main.py', 'r') as f:
    content = f.read()

# Vá 1: f1_score đã bị xóa khỏi torch_geometric bản mới
content = content.replace(
    "from torch_geometric.utils import f1_score, add_self_loops",
    "from torch_geometric.utils import add_self_loops"
)
content = content.replace(
    "train_f1 = torch.mean(f1_score(torch.argmax(y_train.detach(),dim=1), train_target, num_classes=num_classes)).cpu().numpy()",
    "train_f1 = sk_f1_score(train_target.detach().cpu(), torch.argmax(y_train.detach(),dim=1).cpu(), average='macro')"
)
content = content.replace(
    "val_f1 = torch.mean(f1_score(torch.argmax(y_valid,dim=1), valid_target, num_classes=num_classes)).cpu().numpy()",
    "val_f1 = sk_f1_score(valid_target.detach().cpu(), torch.argmax(y_valid,dim=1).cpu(), average='macro')"
)
content = content.replace(
    "test_f1 = torch.mean(f1_score(torch.argmax(y_test,dim=1), test_target, num_classes=num_classes)).cpu().numpy()",
    "test_f1 = sk_f1_score(test_target.detach().cpu(), torch.argmax(y_test,dim=1).cpu(), average='macro')"
)

# Vá 2: đường dẫn ../data -> data
content = content.replace("'../data/%s/", "'data/%s/")

with open('main.py', 'w') as f:
    f.write(content)

# Kiểm tra luôn
with open('main.py') as f:
    ok = 'from torch_geometric.utils import f1_score' not in f.read()
print("Đã vá thành công:", ok)

Đã vá thành công: True


In [ ]:
!python main.py --dataset MovieLens --model FastGTN --num_layers 3 --epoch 200 --lr 0.05 --channel_agg mean --num_channels 2

Namespace(model='FastGTN', dataset='MovieLens', epoch=200, node_dim=64, num_channels=2, lr=0.05, weight_decay=0.001, num_layers=3, runs=10, channel_agg='mean', remove_self_loops=False, non_local=False, non_local_weight=0, beta=0, K=1, pre_train=False, num_FastGTN_layers=1)
/content/model_fastgtn.py:158: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  mat_a = torch.sparse_coo_tensor(a_edge, a_value, (num_nodes, num_nodes)).to(a_edge.device)
Run 0
--------------------Best Result-------------------------
Train - Loss: 0.9514, Macro_F1: 0.7183, Micro_F1: 0.7188
Valid - Loss: 1.1219, Macro_F1: 0.5153, Micro_F1: 0.5538
Test - Loss: 0.9514, Macro_F

In [ ]:
import pickle, torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score

# Load đúng dữ liệu và split đã dùng cho FastGTN
with open('data/MovieLens/node_features.pkl','rb') as f: X = pickle.load(f)
with open('data/MovieLens/labels.pkl','rb') as f: L = pickle.load(f)

# MLP chỉ thấy feature user (giới tính + nghề nghiệp) — KHÔNG thấy đồ thị
X = torch.tensor(X, dtype=torch.float32).cuda()
tr_i, tr_y = torch.tensor(L[0][:,0]).cuda(), torch.tensor(L[0][:,1]).cuda()
va_i, va_y = torch.tensor(L[1][:,0]).cuda(), torch.tensor(L[1][:,1]).cuda()
te_i, te_y = torch.tensor(L[2][:,0]).cuda(), torch.tensor(L[2][:,1]).cuda()

def run_once(seed):
    torch.manual_seed(seed)
    mlp = nn.Sequential(nn.Linear(23,64), nn.ReLU(), nn.Dropout(0.5),
                        nn.Linear(64,64), nn.ReLU(), nn.Dropout(0.5),
                        nn.Linear(64,3)).cuda()
    opt = torch.optim.Adam(mlp.parameters(), lr=0.005, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    best_val, best_test_ma, best_test_mi = 0, 0, 0
    for ep in range(200):
        mlp.train(); opt.zero_grad()
        loss = loss_fn(mlp(X[tr_i]), tr_y)
        loss.backward(); opt.step()
        mlp.eval()
        with torch.no_grad():
            va_pred = mlp(X[va_i]).argmax(1).cpu()
            te_pred = mlp(X[te_i]).argmax(1).cpu()
        val_f1 = f1_score(va_y.cpu(), va_pred, average='macro')
        if val_f1 > best_val:
            best_val = val_f1
            best_test_ma = f1_score(te_y.cpu(), te_pred, average='macro')
            best_test_mi = f1_score(te_y.cpu(), te_pred, average='micro')
    return best_test_ma, best_test_mi

results = [run_once(s) for s in range(10)]
ma = np.array([r[0] for r in results]); mi = np.array([r[1] for r in results])
print(f"MLP baseline (không đồ thị) — Test Macro_F1: {ma.mean():.4f}+{ma.std():.4f}, Micro_F1: {mi.mean():.4f}+{mi.std():.4f}")
print(f"FastGTN (có đồ thị)        — Test Macro_F1: 0.5545+0.0178, Micro_F1: 0.5705+0.0156")

MLP baseline (không đồ thị) — Test Macro_F1: 0.5222+0.0086, Micro_F1: 0.5511+0.0044
FastGTN (có đồ thị)        — Test Macro_F1: 0.5545+0.0178, Micro_F1: 0.5705+0.0156
